# Importation of library

In [ ]:
import tensorflow as tf
print(tf.__version__)

# list of all physical GPUs available on my system
print("Num GPUs Available: ", len(tf.config.list_physical_devices('GPU')))

In [ ]:
# Limit GPU memory usage

# the gpus variable contains a list of all physical GPUs available on my system
gpus = tf.config.experimental.list_physical_devices('GPU')

# This feature sets dynamic GPU memory growth to True, which frees up unused GPU memory.
# By using this feature, you can control the amount of GPU memory used by TensorFlow and avoid GPU out of memory errors.
for gpu in gpus:
   tf.config.experimental.set_memory_growth(gpu, True)

In [ ]:
import sys
sys.path.append("/home/hhousseinho/MIAT/Documents/Srna_Classification")

In [ ]:
import json
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras import backend as K
from tensorflow import keras
from tensorflow.keras.utils import Sequence
from tensorflow.keras import layers
from tensorflow.keras import models
from keras.layers import Input, Conv1D, Conv2D, GlobalMaxPooling1D, GlobalMaxPooling2D, Dropout, Flatten, Dense
from keras.models import Model
from matplotlib import pyplot
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.metrics import precision_score
from sklearn.metrics import recall_score
from sklearn.metrics import f1_score
from sklearn.metrics import cohen_kappa_score
from sklearn.metrics import roc_auc_score
from sklearn.metrics import confusion_matrix
import os 
from scipy.sparse import coo_matrix
from Functions.ReadMatrixParse import ReadMatrixParse
from tensorflow.keras.utils import plot_model
import tensorboard

In [ ]:
# Load the TensorBoard notebook extension.
%load_ext tensorboard

# Load the Data Directory from json file

In [ ]:
with open('/home/hhousseinho/MIAT/Documents/Srna_Classification/Data/data_directory.json', 'r') as f:
    data_directory = json.load(f)
    print(data_directory)

# Data presentation

In [ ]:
TRNA = ReadMatrixParse(data_directory['TRNA_RFAM_Reducted'])
TRNA_RFAM_Reduit = np.stack(TRNA[0], axis=0)
TRNA_RFAM_Reduit.shape

In [ ]:
# Let's plot some arrays of my matrice parse

array = TRNA[0][0:4]

# Then I figured out the number of rows and columns needed to display all the heatmaps by using the "sqrt" function to find the square root of 100,
# then the "ceil" function to round up for the number of columns.

# Determine the number of rows and columns needed to display heatmaps
num_rows = int(np.sqrt(len(array)))
num_cols = int(np.ceil(len(array) / num_rows))

# then I created a figure with the number of subplots needed using the "subplots" function with the "num_rows" and "num_cols" arguments.

# Create a figure with the necessary subgraphs
fig, axs = plt.subplots(num_rows, num_cols, figsize=(15, 15))

# then I used a loop to display each heatmap in the corresponding sub-graph using the "imshow" function and specifying the "hot" colormap. 
# We have also added titles to each sub-graph.

# Display each heatmap in the corresponding subgraph
for i in range(len(array)):
    row_idx = i // num_cols
    col_idx = i % num_cols
    axs[row_idx, col_idx].imshow(array[i], cmap='hot')
    axs[row_idx, col_idx].set_title(f'Heatmap {i+1}')
    
# Remove empty subgraphs
if len(array) < num_rows * num_cols:
    for i in range(len(array), num_rows*num_cols):
        row_idx = i // num_cols
        col_idx = i % num_cols
        fig.delaxes(axs[row_idx, col_idx])

# Add a global title for the figure
fig.suptitle('Multiple Heatmaps')

# adjusted the spacings between sub-graphs using the "tight_layout" function
fig.tight_layout()

# Show figure
plt.show()

In [ ]:
SNORD = ReadMatrixParse(data_directory['SNORD_RFAM'])
SNORD_RFAM = np.stack(SNORD[0], axis=0)
SNORD_RFAM.shape
SNORD[0][0]

In [ ]:
# Let's plot some arrays of my matrice parse

array1 = SNORD[0][0:4]

# Then I figured out the number of rows and columns needed to display all the heatmaps by using the "sqrt" function to find the square root of 100,
# then the "ceil" function to round up for the number of columns.

# Determine the number of rows and columns needed to display heatmaps
num_rows = int(np.sqrt(len(array1)))
num_cols = int(np.ceil(len(array1) / num_rows))

# then I created a figure with the number of subplots needed using the "subplots" function with the "num_rows" and "num_cols" arguments.

# Create a figure with the necessary subgraphs
fig, axs = plt.subplots(num_rows, num_cols, figsize=(15, 15))

# then I used a loop to display each heatmap in the corresponding sub-graph using the "imshow" function and specifying the "hot" colormap. 
# We have also added titles to each sub-graph.

# Display each heatmap in the corresponding subgraph
for i in range(len(array1)):
    row_idx = i // num_cols
    col_idx = i % num_cols
    axs[row_idx, col_idx].imshow(array1[i], cmap='hot')
    axs[row_idx, col_idx].set_title(f'Heatmap {i+1}')
    
# Remove empty subgraphs
if len(array1) < num_rows * num_cols:
    for i in range(len(array1), num_rows*num_cols):
        row_idx = i // num_cols
        col_idx = i % num_cols
        fig.delaxes(axs[row_idx, col_idx])

# Add a global title for the figure
fig.suptitle('Multiple Heatmaps')

# adjusted the spacings between sub-graphs using the "tight_layout" function
fig.tight_layout()

# Show figure
plt.show()

In [ ]:
# concatenate my two dataset
X = np.concatenate((SNORD_RFAM,TRNA_RFAM_Reduit), axis=0)

In [ ]:
# create my target variable
Y=[i*0 for i in range(4385)]
for i in range(4385):
    Y.append(1)

In [ ]:
Y = np.array(Y)
Y.shape

# Data split

In [ ]:
# now, I am going to split my dataset into thwo parts : mainset and testset
x_main,x_test,y_main,y_test=train_test_split(X,Y,test_size=0.20,random_state=123)

In [ ]:
# in this bloc , the code below is the step for split my mainset in two parts 
# whom are trainset and validationset
x_train,x_val,y_train,y_val=train_test_split(x_main,y_main,test_size=0.20,random_state=123)

In [ ]:
print('Original Data shape was : ',X.shape)
print('x_train : ',x_train.shape, 'y_train : ',y_train.shape)
print('x_val : ',x_val.shape, 'y_val : ',y_val.shape)
print('x_test : ',x_test.shape, 'y_test : ',y_test.shape)

# Load hyperparameters from JSON file

In [ ]:
# Load hyperparameters from JSON file
with open('/home/hhousseinho/MIAT/Documents/Srna_Classification/Data/hyperparameters_model.json', 'r') as f:
    hyper_param = json.load(f)
    print(hyper_param)

# Data Generator

In [ ]:
# Now,I am going to create my DataGenerator

In [ ]:
class DataGenerator(Sequence):
    def __init__(self, x_set, y_set, batch_size):
        self.x, self.y = x_set, y_set
        self.batch_size = batch_size
        self.dim = (173, 173, 1)

    def __len__(self):
        return int(np.ceil(len(self.x) / float(self.batch_size)))

    def __getitem__(self, idx):
        batch_x = self.x[idx * self.batch_size:(idx + 1) * self.batch_size]
        batch_x = np.expand_dims(batch_x, axis=-1)  # add extra dimension
        batch_y = self.y[idx * self.batch_size:(idx + 1) * self.batch_size]
        batch_y = np.expand_dims(batch_y, axis=-1)
        return batch_x, batch_y
    
train_gen = DataGenerator(x_train, y_train, hyper_param['batch_size'])
val_gen = DataGenerator(x_val, y_val, hyper_param['batch_size'])
test_gen = DataGenerator(x_test, y_test, hyper_param['batch_size'])

In [ ]:
# to verify the shape of my x_train_gen
print(train_gen[0][0].shape)

# to verify the shape of my y_train_gen
print(train_gen[0][1].shape)

# Build the model

In [ ]:
input_shape = (173, 173, 1)

inputs = keras.Input(shape=input_shape)

x = Conv2D(filters=hyper_param['conv2d_filters'], kernel_size=hyper_param['conv2d_kernel_size'],strides=hyper_param['strides'])(inputs)
x = GlobalMaxPooling2D()(x)
x = Dropout(hyper_param['dropout_rate'])(x)
x = Flatten()(x)
x = Dense(hyper_param['dense_units'], activation='relu')(x)
x = Dropout(hyper_param['dropout_rate'])(x)
outputs = Dense(1, activation='sigmoid')(x)

my_model=Model(inputs, outputs)

In [ ]:
# Define Data Shape

# Compilation of the model
my_model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy',tf.keras.metrics.AUC()])

# Viewing the Model Summary
my_model.summary()

# Plot the model

In [ ]:
plot_model(my_model,show_shapes=True,show_layer_activations=True)

# Train the model

In [ ]:
history = my_model.fit(train_gen,
                    epochs=hyper_param['num_epochs'],
                    validation_data=val_gen,
                    callbacks=[keras.callbacks.TensorBoard(log_dir="/home/hhousseinho/Documents/Srna_Classification/Models/structures")]) 

# Evaluate the model

In [ ]:
# Evaluate the model using accuracy metric

score = my_model.evaluate(test_gen, verbose=0)

print(f'Test loss     : {score[0]:4.4f}')
print(f'Test accuracy : {score[1]:4.4f}')
print(f'Test auc : {score[2]:4.4f}')

In [ ]:
# plot loss during training
pyplot.subplot(211)
pyplot.title('Loss')
pyplot.plot(history.history['loss'], label='train')
pyplot.plot(history.history['val_loss'], label='val')
pyplot.legend()
pyplot.show()
#pyplot.savefig("/home/hhousseinho/MIAT/Documents/Srna_Classification/Results/plot_loss_during_training_RNA_secondary_structure.png")

In [ ]:
# plot accuracy during training
pyplot.subplot(212)
pyplot.title('Accuracy')
pyplot.plot(history.history['accuracy'], label='train')
pyplot.plot(history.history['val_accuracy'], label='val')
pyplot.legend()
pyplot.show()
#pyplot.savefig("/home/hhousseinho/MIAT/Documents/Srna_Classification/Results/plot_accuracy_during_training_RNA_secondary_structure.png")

In [ ]:
# plot auc during training
pyplot.subplot(212)
pyplot.title('AUC')
pyplot.plot(history.history['auc'], label='train')
pyplot.plot(history.history['val_auc'], label='val')
pyplot.legend()
pyplot.show()
#pyplot.savefig("/home/hhousseinho/MIAT/Documents/Srna_Classification/Results/plot_auc_during_training_RNA_secondary_structure.png")

In [ ]:
# Use the predict method of your model to generate predictions on the test set
y_pred = my_model.predict(test_gen)

# Apply a threshold of 0.5 to convert the model's output probabilities to binary predictions
y_pred_binary = (y_pred > 0.5).astype('int32')

errors=[ i for i in range(len(y_test)) if y_pred_binary[i]!=y_test[i] ]

rate_error = len(errors)/len(y_test)

print(f'my model have a rate error of {rate_error:.4f} % when he make prédiction')

In [ ]:
# I will look to some metrics to see the performance of prediction of my model
# So for that I will calculate Accuracy,Precision,Recall and F1 Score

# The scikit-learn metrics API expects a 1D array of actual and predicted values for comparison, 
# therefore, I must reduce the 2D prediction arrays to 1D arrays.

# reduce to 1d array
y_pred1 = y_pred_binary[:, 0]

# accuracy: (tp + tn) / (p + n)
# It measures the rate of correct predictions for all individuals
accuracy = accuracy_score(y_test, y_pred1)
print('Accuracy: %f' % accuracy)

# recall or sensitivity: tp / (tp + fn)
# It makes possible to know the percentage of positives well predicted by my model.
# The higher it is, the more the model maximizes the number of True Positives
recall = recall_score(y_test, y_pred1)
print('Recall: %f' % recall)

# precision tp / (tp + fp)
# The accuracy is quite similar to the recall
# The higher it is, the more the model minimizes the number of False Positives.
precision = precision_score(y_test, y_pred1)
print('Precision: %f' % precision)

# f1 score: 2 tp / (2 tp + fp + fn)
# it is a metric allowing to combine precision and recall
# The F1 Score allows us to make a good evaluation of the performance of my model
f1 = f1_score(y_test, y_pred1)
print('F1 score: %f' % f1)

In [ ]:
# I can also calculate some additional metrics, 
# such as the Cohen’s kappa, ROC AUC, and confusion matrix.

# Cohen’s kappa
# generally used to measure the performance of the classifier by comparing it to that of a random classifier.
kappa = cohen_kappa_score(y_test, y_pred1)
print('Cohens kappa: %f' % kappa)

# ROC AUC
# it makes it possible to describe the performance of a model through two indicators: sensitivity and specificity.
auc = roc_auc_score(y_test, y_pred1)
print('ROC AUC: %f' % auc)

# confusion matrix
# it compares actual values to values predicted by the model
matrix = confusion_matrix(y_test, y_pred1)
print(matrix)

# Tensorboard

In [ ]:
%tensorboard --logdir=/home/hhousseinho/Documents/Srna_Classification/Models/structures